# Tutorial 05: The RL Meta-Controller (Halt vs. Continue)

In this final tutorial, we reach the "Top Level" of the LMCOS system. We've seen how to build trees, flatten them, and process them with a GNN. Now, we use that processing to make a **Control Decision**.

### The Question
Given the current state of my search tree, should I **Halt** (play the move now) or **Continue** (spend more energy expanding the tree)?

### The Trade-off
1.  **Benefit of Continuing:** The GNN might discover a better move, increasing our expected game reward.
2.  **Cost of Continuing ($C$):** Every "thought" (expansion) has a constant penalty. If the expected gain is < $C$, it is mathematically optimal to stop.

In [ ]:
import os
import sys
from pathlib import Path
import torch
import numpy as np

# Setup Path
os.environ["PATH"] += os.pathsep + "/opt/homebrew/bin"
LMCOS_DIR = Path(os.getcwd()).parent
if str(LMCOS_DIR) not in sys.path:
    sys.path.insert(0, str(LMCOS_DIR))

from GNN import TreeNN, HaltController
from tensorizer import TreeTensorizer
from schema import tree_encoder_feature_schema
from helper_tensorization import build_demo_trees
from helper_gnn import visualize_halt_decision

# Setup GNN Foundations
schema = tree_encoder_feature_schema()
tensorizer = TreeTensorizer(schema)
trees = build_demo_trees()
batch = tensorizer.tensorize_forest([trees[0]]) # Breadth-first fork

gnn = TreeNN(k=1, node_feat=5, device='cpu', d_embed=8)
halt_head = HaltController(d_embed=8, hidden_dim=64, device='cpu')

## Step 1: The Root Summary ($h_{root}$)

The RL agent doesn't look at the whole tree. It looks at the **Root**. Because of our bidirectional topological sweep, the root vector $h_{root}$ already contains the summarized "presents" and "future potential" of the entire search structure.

In [ ]:
# Run GNN forward pass to get the root state
output = gnn(batch)
h_root = output.root_states[0] # The 8-dimensional summary of Tree 1

print(f"Root Hidden State: {h_root.detach().numpy().round(3)}")

## Step 2: The Halt Decision

We pass that summary into the `HaltController`. This MLP maps the high-dimensional chess context into a single probability: $P(\text{halt})$.

In [ ]:
logits, prob = halt_head(h_root.unsqueeze(0))
prob_val = prob.item()

print(f"Halt Probability: {prob_val:.4f}")
display(visualize_halt_decision(h_root, prob_val, thinking_cost=0.01))

## Step 3: Normative Ground Truth (The DP Oracle)

During training, $P(\text{halt})$ is optimized against an **Oracle Policy**. 

If we have a full sequence of a search from $T_0 \dots T_{64}$, we calculate the optimal halting step $K^*$ back-to-front:
- **If I halt at step 64:** My reward is $V(T_{64})$.
- **If I am at step 63:** Better to halt and get $V(T_{63})$... or continue and get $V(T_{64}) - C$?

This creates a supervised "Reward-to-Go" signal that teaches the model exactly when the cost of thinking outweighs the value of discovery.

## Conclusion: Final Architecture Sync

You have now seen the complete pipeline:
1.  **Tutorial 01-02:** Prepare raw chess search data into flat GPU tensors.
2.  **Tutorial 03-04:** Train a Bidirectional GNN to summarize these trees into predictive latent vectors ($h$).
3.  **Tutorial 05:** Use the GNN's root summary to make optimal, cost-aware decisions about the economy of thought.

This Meta-Controller allows `lmcos` to act as a **Dynamic Intelligence Accelerator**, spending more brain power on complex tactical blunders and skipping trivial forced moves.